[<< Sommaire QC](../README.md) | [Précédent : QC-Py-41-PaperTrading-IBKR <<](./QC-Py-41-PaperTrading-IBKR.ipynb)

# QC-Py-42 - Alpha mining par evolution

**Boucle from-scratch d'evolution d'expressions alpha**, et pont entre la serie `Search/`
(algorithmes evolutionnaires) et la serie `QuantConnect` (validation financiere).

## Pourquoi ce carnet

Le depot mine ses alphas **a la main**. La litterature recente automatise cette etape par
**evolution d'expressions** : R&D-Agent(Q), Microsoft Research Asia, arXiv:2505.15155, archive au
gisement bibliographique `Trading/`. Le depot enseigne deja les algorithmes evolutionnaires
(`deap`, `Search-05-GeneticAlgorithms.ipynb`) mais ne les a **jamais** appliques a la finance.

Mesure de l'angle mort au 2026-10-09, grep prose (`*.py` / `*.md` / `*.yml` / `*.cs`, sorties
`.ipynb` exclues) : `RD-Agent` **0**, `AlphaGen` **0**, `alpha mining` **0**.

## Ce que ce carnet fait

1. Construit un panier crypto depuis les donnees **locales** du depot (10 actifs) : aucune
   dependance reseau, l'execution est reproductible. La plage **2018-2026** est celle des
   fichiers ; la fenetre reellement exploitee, apres intersection des dates communes, est
   **mesuree en §1** et vaut environ cinq ans - c'est elle qui alimente tous les chiffres.
2. Evolue des **expressions alpha** (arbres d'operateurs sur des features de base) avec `deap`.
3. Confronte le meilleur individu a une **baseline de facteurs fixes** (momentum, reversal,
   volatilite, volume) - le temoin negatif sans lequel « l'evolution gagne » ne veut rien dire.
4. Mesure hors-echantillon en **walk-forward a 5 blocs**, sur **4 seeds** (0 / 1 / 7 / 42), avec un
   test de **Diebold-Mariano** sur une perte de precision (`mse`) **et un critere directionnel**
   explicite : une difference significative mais **defavorable** n'est pas une victoire.
5. Rend un **verdict honnete** : `BEATS` / `NO BEATS` / `INCONCLUSIVE`.

> La jambe « mini-boucle LLM » (proposer / executer / apprendre, facon R&D-Agent reduite) est
> **hors de ce carnet** : elle demande un acces modele et un budget d'evaluation distincts.
> Elle est suivie par #20041.

**Duree d'execution** : quelques minutes sur CPU (population et generations volontairement
bornees pour un carnet pedagogique).

## 1. Chargement du panier

Les donnees viennent du depot (`QuantConnect/datasets/yfinance/crypto_panier/`). On ne fait
**aucun forward-fill** : un trou de cotation n'est pas un prix. L'intersection des dates communes
est donc prise telle quelle, ce qui raccourcit la periode au plus tardif des demarrages.

In [1]:
import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)

SEEDS = (0, 1, 7, 42)                 # plancher multi-seed de la doctrine §C
ASSETS = ["BTC-USD", "ETH-USD", "ADA-USD", "AVAX-USD", "DOT-USD",
          "LINK-USD", "LTC-USD", "MATIC-USD", "SOL-USD", "XRP-USD"]


def find_panier_dir():
    here = os.path.abspath(os.getcwd())
    for _ in range(7):
        cand = os.path.join(here, "MyIA.AI.Notebooks", "QuantConnect",
                            "datasets", "yfinance", "crypto_panier")
        if os.path.isdir(cand):
            return cand
        here = os.path.dirname(here)
    raise FileNotFoundError("panier introuvable depuis %s" % os.getcwd())


PANIER = find_panier_dir()
print("panier   :", PANIER)
print("csv vus  :", len(glob.glob(os.path.join(PANIER, "*.csv"))))

panier   : D:\dev\CoursIA-20041-alpha\MyIA.AI.Notebooks\QuantConnect\datasets\yfinance\crypto_panier
csv vus  : 10


In [2]:
frames = {}
for a in ASSETS:
    p = os.path.join(PANIER, "%s_2018-01-01_2026-05-01.csv" % a)
    if os.path.exists(p):
        frames[a] = pd.read_csv(p, parse_dates=["Date"]).set_index("Date").sort_index()

close = pd.DataFrame({a: f["Close"] for a, f in frames.items()}).sort_index().dropna(how="any")
volume = pd.DataFrame({a: f["Volume"] for a, f in frames.items()}).sort_index().reindex(close.index)

print("periode  :", close.index.min().date(), "->", close.index.max().date())
print("forme    :", close.shape, "(jours x actifs)")
print("actifs   :", list(close.columns))
close.tail(3)

periode  : 2020-09-22 -> 2025-03-24
forme    : (1645, 10) (jours x actifs)
actifs   : ['BTC-USD', 'ETH-USD', 'ADA-USD', 'AVAX-USD', 'DOT-USD', 'LINK-USD', 'LTC-USD', 'MATIC-USD', 'SOL-USD', 'XRP-USD']


,BTC-USD,ETH-USD,ADA-USD,AVAX-USD,DOT-USD,LINK-USD,LTC-USD,MATIC-USD,SOL-USD,XRP-USD
Date,,,,,,,,,,
2025-03-22,83832.484375,1980.037842,0.700941,19.420124,4.444770,14.226722,91.257545,0.209941,128.484985,2.369824
2025-03-23,86054.375000,2005.299194,0.709508,19.811661,4.495940,14.435690,91.415352,0.212728,132.599319,2.435493
2025-03-24,87498.914062,2077.479004,0.730995,21.460428,4.616263,15.089212,93.674507,0.216415,140.869278,2.448162


### Rendements et cible

L'alpha cherche a predire le **rendement transversal du lendemain**. On construit donc :

- `ret` : rendement quotidien simple ;
- `fwd` : cible = `ret` decale d'un jour (ce que l'on cherche a predire).

Le decalage est la seule precaution anti-fuite : aucune feature ne doit voir le jour qu'elle predit.

In [3]:
ret = close.pct_change().replace([np.inf, -np.inf], np.nan)
vol = volume.replace(0, np.nan)

# Cible : rendement du lendemain. shift(-1) = anti-fuite.
fwd = ret.shift(-1)

valid = ret.notna().all(axis=1) & fwd.notna().all(axis=1)
# close est filtre EN MEME TEMPS que ret/vol/fwd. Sans cela, les terminaux batis sur
# close (logclose) gardent les lignes que ret a retirees, et toute expression qui
# melange les deux familles est rejetee par le controle de forme d'evaluate().
close, ret, vol, fwd = close[valid], ret[valid], vol[valid], fwd[valid]

print("jours exploitables :", len(ret))
print("rendement quotidien moyen (transversal) : %.5f" % ret.mean().mean())
ret.describe().loc[["mean", "std"]].round(5)

jours exploitables : 1643
rendement quotidien moyen (transversal) : 0.00250


,BTC-USD,ETH-USD,ADA-USD,AVAX-USD,DOT-USD,LINK-USD,LTC-USD,MATIC-USD,SOL-USD,XRP-USD
mean,0.00179,0.00194,0.00271,0.00283,0.00146,0.00179,0.00154,0.00353,0.00433,0.00310
std,0.03210,0.04153,0.05405,0.06480,0.05351,0.05455,0.04678,0.06664,0.06364,0.05967


## 2. Operateurs, terminaux, et evaluation d'une expression

Une **expression alpha** est un arbre dont :

- les **feuilles** sont des features de base (prix, volume, rendement) ;
- les **noeuds** sont des operateurs (moyenne glissante, ecart-type glissant, delta, rang
  transversal, negation, arithmetique protegee).

`deap` fournit les operateurs evolutionnaires ; notre travail est de definir le **jeu de
primitives** et la **fonction de fitness**. C'est exactement le pont Search -> QuantConnect qui
n'existait pas.

Les operateurs temporels prennent une **fenetre** en parametre ; on expose donc des variantes
nommees (`mean5`, `mean20`, ...) plutot qu'un argument libre, ce qui garde l'arbre typable et
l'espace de recherche interprettable.

In [4]:
def zscore_rows(x):
    mu = np.nanmean(x, axis=1, keepdims=True)
    sd = np.nanstd(x, axis=1, keepdims=True)
    return (x - mu) / np.where(sd == 0, np.nan, sd)


def rank_rows(x):
    out = np.full_like(x, np.nan, dtype=float)
    for i in range(x.shape[0]):
        row = x[i]
        ok = ~np.isnan(row)
        if ok.sum() < 2:
            continue
        order = np.argsort(np.argsort(row[ok]))
        out[i, ok] = order / (ok.sum() - 1.0)
    return out


def ts_mean(x, w):
    return pd.DataFrame(x).rolling(w, min_periods=max(2, w // 2)).mean().values


def ts_std(x, w):
    return pd.DataFrame(x).rolling(w, min_periods=max(2, w // 2)).std().values


def ts_delta(x, w):
    d = pd.DataFrame(x)
    return (d - d.shift(w)).values


def ts_rank_ref(x, w):
    # Reference pandas : lisible, mais 1,7 s par appel sur un panneau (1643 x 10)
    # (rolling.apply passe par une boucle Python). Inutilisable dans une boucle
    # evolutionnaire qui evalue des milliers d'expressions -- gardee ici comme
    # oracle de la version vectorisee ci-dessous.
    return pd.DataFrame(x).rolling(w, min_periods=max(2, w // 2)).apply(
        lambda r: (r.iloc[-1] > r).mean(), raw=False).values


def ts_rank(x, w):
    # Rang (0..1) de la derniere valeur dans sa fenetre glissante de w jours,
    # en version vectorisee : 0,020 s au lieu de 1,7 s, meme resultat au bit.
    # Deux details de la semantique pandas sont reproduits : (a) pendant
    # l'amorcage, la fenetre est plus courte que w et min_periods vaut w // 2 ;
    # (b) les NaN de la fenetre comptent dans le denominateur (pandas passe la
    # fenetre brute, pas la fenetre nettoyee).
    from numpy.lib.stride_tricks import sliding_window_view
    T, N = x.shape
    m = max(2, w // 2)
    out = np.full((T, N), np.nan)
    for i in range(m - 1, T):
        lo = max(0, i - w + 1)
        win = x[lo:i + 1, :]
        ok = ~np.isnan(win)
        cnt = ok.sum(axis=0)
        gt = ((x[i, :] > win) & ok).sum(axis=0)
        out[i, :] = np.where(cnt >= m, gt / np.float64(i - lo + 1), np.nan)
    return out


def sdiv(a, b):
    return np.where(np.abs(b) < 1e-12, np.nan, a / b)


def neg(x):
    return -x


def slog(x):
    return np.sign(x) * np.log1p(np.abs(x))


PRIMITIVES = {
    "add": (np.add, 2), "sub": (np.subtract, 2), "mul": (np.multiply, 2),
    "div": (sdiv, 2), "neg": (neg, 1), "log": (slog, 1),
    "rank": (rank_rows, 1), "zscore": (zscore_rows, 1),
    "mean5": (lambda x: ts_mean(x, 5), 1), "mean20": (lambda x: ts_mean(x, 20), 1),
    "std5": (lambda x: ts_std(x, 5), 1), "std20": (lambda x: ts_std(x, 20), 1),
    "delta5": (lambda x: ts_delta(x, 5), 1), "delta20": (lambda x: ts_delta(x, 20), 1),
    "tsrank20": (lambda x: ts_rank(x, 20), 1),
}

TERMINALS = {
    "logclose": np.log(close.values),
    "ret": ret.values,
    "vol": np.log1p(vol.values),
}

print("primitives :", len(PRIMITIVES))
print("terminaux  :", list(TERMINALS))

# ts_rank est vectorise pour tenir dans la boucle evolutionnaire. L'equivalence
# avec l'oracle pandas est MESUREE ici, sur les donnees reelles, et non supposee :
# la ligne imprimee doit afficher 0.0.
_r = ts_rank_ref(ret.values, 20)
_f = ts_rank(ret.values, 20)
_common = np.isfinite(_r) & np.isfinite(_f)
print("ts_rank : ecart max vs oracle pandas = %.3g  (masques identiques : %s)"
      % (np.nanmax(np.abs(_r[_common] - _f[_common])),
         np.array_equal(np.isfinite(_r), np.isfinite(_f))))

primitives : 15
terminaux  : ['logclose', 'ret', 'vol']


ts_rank : ecart max vs oracle pandas = 0  (masques identiques : True)


### Interpretreur d'expression

L'expression est un arbre en notation prefixee (`list`) : `['div', ['delta5', ['logclose']], ['std20', ['ret']]]`.
Un interpreteur recursif l'evalue sur le panneau complet. Evaluer l'arbre soi-meme (plutot que
`deap.gp.compile`) permet de **rejeter proprement** une expression qui produit `NaN` partout, au
lieu de propager une exception au milieu de la boucle evolutionnaire.

In [5]:
def evaluate(expr, terminals=TERMINALS):
    # Evalue une expression prefixee -> ndarray (T, N). None si non evaluable.
    if isinstance(expr, str):
        return terminals[expr]
    head = expr[0]
    if head not in PRIMITIVES:
        return None
    fn, arity = PRIMITIVES[head]
    args = []
    for k in range(1, arity + 1):
        v = evaluate(expr[k], terminals)
        if v is None:
            return None
        args.append(v)
    try:
        out = fn(*args)
    except Exception:
        return None
    out = np.asarray(out, dtype=float)
    if out.shape != (len(close), close.shape[1]):
        return None
    return out


def expression_size(expr):
    if isinstance(expr, str):
        return 1
    return 1 + sum(expression_size(e) for e in expr[1:])


def to_prefix(expr):
    if isinstance(expr, str):
        return expr
    return "(%s %s)" % (expr[0], " ".join(to_prefix(e) for e in expr[1:]))


# Convention d'ecriture : un TERMINAL est une chaine nue ("logclose"), jamais une
# liste a un element (["logclose"]) -- une liste est un noeud d'operateur, et
# ["logclose"] serait alors lu comme un appel a une primitive inexistante.
demo = ["div", ["delta5", "logclose"], ["std20", "ret"]]
d = evaluate(demo)
print("expression :", to_prefix(demo))
print("forme      :", None if d is None else d.shape)
print("fini       :", None if d is None else bool(np.isfinite(d).any()))

expression : (div (delta5 logclose) (std20 ret))
forme      : (1643, 10)
fini       : True


### Fonction de fitness

Le fitness est l'**IC transversal moyen** (correlation de rang, jour par jour, entre signal et
rendement du lendemain) sur la **fenetre d'entrainement** uniquement. Un IC moyen eleve et stable
est ce qu'un alpha doit produire.

Le fitness n'est **jamais** calcule sur la fenetre de test : c'est ce qui rend le verdict
hors-echantillon credible.

In [6]:
def signal_to_rank(sig):
    return pd.DataFrame(sig).rank(axis=1, pct=True).values


def daily_ic(sig, target):
    s = signal_to_rank(sig)
    t = pd.DataFrame(target).rank(axis=1, pct=True).values
    both = np.isfinite(s) & np.isfinite(t)
    n = both.sum(axis=1)
    keep = n >= 5
    if keep.sum() < 20:
        return np.nan
    s, t = s[keep], t[keep]
    sm = s - s.mean(axis=1, keepdims=True)
    tm = t - t.mean(axis=1, keepdims=True)
    num = (sm * tm).sum(axis=1)
    den = np.sqrt((sm ** 2).sum(axis=1) * (tm ** 2).sum(axis=1))
    ic = np.where(den == 0, np.nan, num / den)
    return float(np.nanmean(ic))


def long_short_daily(sig, target, q=0.3):
    # Rendement quotidien d'un portefeuille long top-q / short bottom-q, equipondere.
    s = signal_to_rank(sig)
    k = max(1, int(round(s.shape[1] * q)))
    out = np.full(s.shape[0], np.nan)
    for i in range(s.shape[0]):
        row, tgt = s[i], target[i]
        ok = np.isfinite(row) & np.isfinite(tgt)
        if ok.sum() < 2 * k:
            continue
        idx = np.argsort(row[ok])
        labs = np.array(tgt[ok])[idx]
        out[i] = labs[-k:].mean() - labs[:k].mean()
    return out


def sharpe(daily):
    d = daily[np.isfinite(daily)]
    if len(d) < 20 or d.std() == 0:
        return np.nan
    return float(d.mean() / d.std() * np.sqrt(365))

## 3. Baseline - les facteurs fixes

Le **temoin negatif**. Quatre facteurs classiques, chacun un signal connu :

| Facteur | Signal |
|---|---|
| momentum | rendement cumule sur 20 jours |
| reversal | negation du rendement de la veille |
| volatilite | ecart-type glissant 20 jours, negatif (low-vol) |
| volume | rang transversal du volume |

Si l'evolution ne bat pas ces facteurs, le verdict est `NO BEATS` - et c'est un resultat utile.

### Le decoupage, et ce que chaque etape lit

La serie est coupee en **6 intervalles** contigus, qui forment **5 folds** : le fold *i*
entraine sur l'intervalle *i* et teste sur le suivant. Les 6 intervalles sont donc lus, sans
reste : c'est le decoupage **walk-forward a 5 blocs** que la doctrine §C demande (§C en exige
cinq ; un decoupage a 4 folds laissait le dernier intervalle hors de toute lecture).

La **baseline** est caracterisee sur les 5 folds. Le **champion evolue**, lui, est entraine
**une seule fois** (sur l'intervalle d'entrainement du fold 0) puis re-evalue sur les blocs de
test des folds **1 a 4** - il ne peut pas etre re-entraine par fold, sinon chaque fold aurait
son propre champion et l'ecart inter-seeds ne mesurerait plus la dispersion d'une meme methode.

La comparaison finale porte donc sur **la meme fenetre pour les deux** : la colonne
`Sharpe OOS (comparaison)` de la table ci-dessous restreint la baseline a ces quatre blocs, ce
qui la rend homometrique du champion. Comparer le champion (4 blocs) a une baseline moyennee
sur 5 blocs comparerait deux fenetres differentes - l'ecart ne voudrait rien dire.

Enfin, le **facteur de reference** est choisi sur l'**IC d'entrainement** (colonne
`IC train moyen`), jamais sur le Sharpe hors-echantillon de la fenetre de comparaison :
selectionner la reference sur la fenetre meme ou l'on compare reviendrait a choisir le
meilleur adversaire *apres* avoir vu le resultat.

In [7]:
FIXED = {
    "momentum": evaluate(["delta20", "logclose"]),
    "reversal": evaluate(["neg", "ret"]),
    "lowvol": evaluate(["neg", ["std20", "ret"]]),
    "volume": evaluate(["rank", "vol"]),
}
if any(v is None for v in FIXED.values()):
    raise RuntimeError("facteur fixe non evaluable : %s"
                       % [k for k, v in FIXED.items() if v is None])

# Decoupage walk-forward a 5 blocs : 6 intervalles contigus, tous consommes
# (la doctrine §C exige 5 folds ; a 4 folds le dernier intervalle n'etait lu par personne).
n = len(ret)
edges = np.linspace(0, n, 7).astype(int)
FOLDS = [(edges[i], edges[i + 1], edges[i + 1], edges[i + 2]) for i in range(5)]

# Le champion est entraine UNE fois sur FOLDS[0] puis re-evalue sur les blocs de test
# de FOLDS[1:] : ce sont ces memes blocs qui portent la comparaison baseline <-> champion,
# d'ou EVAL_FOLDS, utilise aussi bien dans la table ci-dessous qu'a la section 5.
EVAL_FOLDS = FOLDS[1:]

rows = []
for name, sig in FIXED.items():
    if sig is None:
        continue
    ics, shps, shps_eval = [], [], []
    for tr0, tr1, te0, te1 in FOLDS:
        ics.append(daily_ic(sig[tr0:tr1], fwd.values[tr0:tr1]))
        shps.append(sharpe(long_short_daily(sig[te0:te1], fwd.values[te0:te1])))
    for tr0, tr1, te0, te1 in EVAL_FOLDS:
        shps_eval.append(sharpe(long_short_daily(sig[te0:te1], fwd.values[te0:te1])))
    rows.append({"facteur": name,
                 "IC train moyen": np.nanmean(ics),
                 "Sharpe OOS (5 blocs)": np.nanmean(shps),
                 "Sharpe OOS (comparaison)": np.nanmean(shps_eval)})

baseline = pd.DataFrame(rows).set_index("facteur").round(4)
print("baseline facteurs fixes (walk-forward %d blocs ; comparaison sur %d blocs)"
      % (len(FOLDS), len(EVAL_FOLDS)))
# Reference choisie sur l'IC d'ENTRAINEMENT : la ligne de comparaison est celle du facteur
# retenu, visible ici pour que le lecteur puisse verifier qu'il n'a pas ete choisi apres coup.
baseline.sort_values("IC train moyen", ascending=False)

baseline facteurs fixes (walk-forward 5 blocs ; comparaison sur 4 blocs)


,IC train moyen,Sharpe OOS (5 blocs),Sharpe OOS (comparaison)
facteur,,,
lowvol,0.0586,-0.8184,-0.6359
volume,0.0509,0.2739,1.0793
reversal,0.0199,-0.8939,-0.8912
momentum,0.0102,1.6890,1.5247


## 4. Evolution d'expressions avec `deap`

On evolue des arbres d'expressions. Le fitness est l'IC transversal sur la **fenetre
d'entrainement** (`eaMuPlusLambda`, selection par tournoi). Chaque seed est une execution
independante : c'est le plancher **multi-seed ≥ 4** de la doctrine §C.

La taille de l'arbre est **bornee** : un alpha doit rester lisible, et une expression obeese
sur-ajuste.

In [8]:
import random
from deap import base, creator, tools, algorithms

MAX_DEPTH = 3
POP, NGEN, LAMBDA = 60, 8, 60

PRIM_NAMES = list(PRIMITIVES)
TERM_NAMES = list(TERMINALS)


def random_expr(rng, depth=0):
    if depth >= MAX_DEPTH or (depth > 0 and rng.random() < 0.35):
        return rng.choice(TERM_NAMES)
    name = rng.choice(PRIM_NAMES)
    arity = PRIMITIVES[name][1]
    return [name] + [random_expr(rng, depth + 1) for _ in range(arity)]


def mutate_expr(expr, rng):
    if rng.random() < 0.3:
        return random_expr(rng)
    if isinstance(expr, str):
        return rng.choice(TERM_NAMES)
    idx = rng.randrange(1, len(expr))
    expr = list(expr)
    expr[idx] = mutate_expr(expr[idx], rng)
    return expr


if not hasattr(creator, "FitnessMaxAlpha"):
    creator.create("FitnessMaxAlpha", base.Fitness, weights=(1.0,))
if not hasattr(creator, "IndividualAlpha"):
    creator.create("IndividualAlpha", list, fitness=creator.FitnessMaxAlpha)


def evolution(tr0, tr1, seed):
    # Fait evoluer des expressions sur la fenetre d'entrainement [tr0, tr1). Rend le meilleur prefixe.
    # evaluate() rend le signal sur TOUTE la serie : la fitness doit le decouper sur la
    # meme fenetre que la cible, sinon daily_ic compare (T, N) a (tr1-tr0, N).
    rng = random.Random(seed)
    # DEAP puise dans le module `random` GLOBAL, pas dans notre generateur local :
    # la selection (`selTournament`, qui ne prend aucun argument `rng` en 1.4) ET le
    # choix du parent a muter (`algorithms.varOr` -> `random.choice`). Sans ce seed,
    # deux executions du meme carnet rendaient des champions differents -- mesure sur
    # la fenetre d'entrainement, seed 0 : Sharpe OOS 0.1304 puis 0.6912 -- et la
    # validation multi-seed ne voulait plus rien dire. Le seed global est repose a
    # chaque appel, donc chaque seed de SEEDS reste independant des autres.
    random.seed(seed)
    target = fwd.values[tr0:tr1]

    def fit(individual):
        v = evaluate(individual)
        if v is None:
            return (-1.0,)
        ic = daily_ic(v[tr0:tr1], target)
        if ic is None or not np.isfinite(ic):
            return (-1.0,)
        return (float(ic) - 0.001 * expression_size(individual),)

    tb = base.Toolbox()
    tb.register("individual", tools.initIterate, creator.IndividualAlpha,
                lambda: random_expr(rng))
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("evaluate", fit)
    tb.register("mate", lambda a, b: (a, b))
    # La mutation doit rendre un IndividualAlpha, pas une list nue : un descendant
    # de type list perd son attribut .fitness et selTournament leve
    # "AttributeError: 'list' object has no attribute 'fitness'". Le noeud racine
    # reste une liste (mutate_expr ne remplace qu'un sous-noeud d'indice >= 1).
    tb.register("mutate",
                lambda ind: (creator.IndividualAlpha(mutate_expr(list(ind), rng)),))
    tb.register("select", tools.selTournament, tournsize=3)

    pop = tb.population(n=POP)
    hof = tools.HallOfFame(1)
    algorithms.eaMuPlusLambda(pop, tb, mu=POP, lambda_=LAMBDA,
                              cxpb=0.0, mutpb=1.0, ngen=NGEN,
                              halloffame=hof, verbose=False)
    return hof[0]


t0 = time.time()
champions = {}
for seed in SEEDS:
    tr0, tr1, te0, te1 = FOLDS[0]
    best = evolution(tr0, tr1, seed)
    champions[seed] = best
    v = evaluate(best)
    ic = daily_ic(v[tr0:tr1], fwd.values[tr0:tr1]) if v is not None else float("nan")
    print("seed %-3d IC train %.4f  taille %2d  %s" % (
        seed, ic, expression_size(best), to_prefix(best)[:90]))
print("duree : %.1f s" % (time.time() - t0))

seed 0   IC train 0.1273  taille  7  (zscore (mean20 (div (tsrank20 ret) (rank logclose))))


seed 1   IC train 0.0711  taille  9  (div (div (rank logclose) (std20 ret)) (tsrank20 (std20 vol)))


seed 7   IC train 0.0956  taille  4  (mean20 (neg (std20 ret)))


seed 42  IC train 0.0835  taille  7  (zscore (mul vol (div vol (std20 ret))))
duree : 34.3 s


## 5. Evaluation hors-echantillon et Diebold-Mariano

Le champion de chaque seed est **re-evalue sur les blocs de test**, qu'il n'a jamais vus. On
mesure :

- l'**IC hors-echantillon** ;
- le **Sharpe** du portefeuille long-short ;
- un test de **Diebold-Mariano** sur une perte de precision (`mse`) : le modele evolue est
  confronte au **meilleur facteur fixe**, jour par jour, sur la meme cible.

Le champion est entraine **une seule fois** (intervalle d'entrainement du fold 0) et les blocs de
test qu'il rencontre ici sont **exactement** ceux qui portent la valeur de comparaison de la
baseline : la colonne `Sharpe OOS (comparaison)` de la section 3. Sans cette homometrie, on
comparerait une moyenne sur 4 blocs a une moyenne sur 5.

La doctrine §C demande la **conjonction** de **trois** choses :

1. un ecart inter-seeds significatif (≥ 2 sigma) ;
2. `dm_p_median < 0.05` ;
3. un **signe favorable** de la statistique de Diebold-Mariano.

Le troisieme point est celui qui manquait. La statistique est
`moyenne(perte_evolue - perte_fixe) / se` : **positive, elle dit que l'evolue est MOINS precis**.
`p < 0.05` seul recompenserait donc une difference **significative et defavorable** - c'est-a-dire
une defaite nette - exactement comme une victoire. La cellule 5.3 ferme ce trou et le prouve par
un **temoin negatif** execute : deux series de pertes ou l'evolue est significativement moins
precis, que l'ancien critere acceptait et que le nouveau refuse.

In [9]:
# Diebold-Mariano implemente localement (Newey-West HAC, lag 1) : aucune dependance exotique.
def dm_test_hac(loss_a, loss_b, lag=1):
    d = np.asarray(loss_a, float) - np.asarray(loss_b, float)
    d = d[np.isfinite(d)]
    n = len(d)
    if n < 30:
        return np.nan, np.nan
    mean_d = d.mean()
    dc = d - mean_d
    gamma0 = (dc ** 2).sum() / n
    s = gamma0
    for k in range(1, lag + 1):
        gk = (dc[k:] * dc[:-k]).sum() / n
        s += 2.0 * (1.0 - k / (lag + 1.0)) * gk
    se = np.sqrt(s / n)
    if se == 0:
        return np.nan, np.nan
    stat = mean_d / se
    from scipy import stats as st
    p = 2 * (1 - st.t.cdf(abs(stat), df=n - 1))
    return float(stat), float(p)


def oos_losses(sig, target):
    # Perte de precision (mse) du signal pris comme predicteur du rendement du lendemain.
    s = signal_to_rank(sig)
    pred = (s - 0.5) * 2.0
    err = pred - pd.DataFrame(target).rank(axis=1, pct=True).values
    return np.nanmean(err ** 2, axis=1)


# Reference choisie sur l'IC d'ENTRAINEMENT, pas sur le Sharpe OOS de la fenetre de
# comparaison : choisir le meilleur facteur sur la fenetre ou l'on compare reviendrait a
# designer l'adversaire apres avoir vu le score. La table ci-dessus rend les deux colonnes
# visibles pour que le lecteur puisse juger le choix.
best_fixed_name = baseline["IC train moyen"].idxmax()
best_fixed = FIXED[best_fixed_name]
print("facteur fixe de reference (choisi sur l'IC d'entrainement) :", best_fixed_name)
print("  (pour transparence, Sharpe OOS de comparaison des 4 facteurs :)")
print(baseline["Sharpe OOS (comparaison)"].to_string())

rows = []
for seed in SEEDS:
    sig = evaluate(champions[seed])
    ics, shps, losses = [], [], []
    for tr0, tr1, te0, te1 in EVAL_FOLDS:
        ics.append(daily_ic(sig[te0:te1], fwd.values[te0:te1]))
        shps.append(sharpe(long_short_daily(sig[te0:te1], fwd.values[te0:te1])))
        losses.append(oos_losses(sig[te0:te1], fwd.values[te0:te1]))
    rows.append({"seed": seed, "IC OOS": np.nanmean(ics),
                 "Sharpe OOS": np.nanmean(shps),
                 "_loss": np.concatenate(losses)})

evolved = pd.DataFrame([{k: v for k, v in r.items() if k != "_loss"} for r in rows])

fixed_loss = np.concatenate([oos_losses(best_fixed[te0:te1], fwd.values[te0:te1])
                             for _, _, te0, te1 in EVAL_FOLDS])

dm_stats = []
for r in rows:
    L = min(len(r["_loss"]), len(fixed_loss))
    dm_stats.append(dm_test_hac(r["_loss"][:L], fixed_loss[:L]))
evolved["DM stat"] = [s for s, _ in dm_stats]
evolved["DM p"] = [p for _, p in dm_stats]
evolved = evolved.set_index("seed").round(4)

print("\nchampion evolue, hors-echantillon (walk-forward, %d blocs)" % len(EVAL_FOLDS))
evolved

facteur fixe de reference (choisi sur l'IC d'entrainement) : lowvol
  (pour transparence, Sharpe OOS de comparaison des 4 facteurs :)
facteur
momentum    1.5247
reversal   -0.8912
lowvol     -0.6359
volume      1.0793



champion evolue, hors-echantillon (walk-forward, 4 blocs)


,IC OOS,Sharpe OOS,DM stat,DM p
seed,,,,
0,-0.1192,-0.2013,3.7261,0.0002
1,0.0347,0.1523,-4.6425,0.0000
7,0.0383,-0.4268,-0.0018,0.9986
42,0.0577,-0.2530,-3.6436,0.0003


In [10]:
edge = evolved["Sharpe OOS"].mean()
sigma = evolved["Sharpe OOS"].std(ddof=1)
# Meme fenetre que `edge` : la colonne de comparaison restreint la baseline aux blocs de
# test de EVAL_FOLDS, ceux-la memes que le champion vient de parcourir.
fixed_sharpe = baseline.loc[best_fixed_name, "Sharpe OOS (comparaison)"]
dm_p_median = float(np.nanmedian(evolved["DM p"]))
dm_stat_median = float(np.nanmedian(evolved["DM stat"]))

sigma_ok = np.isfinite(sigma) and sigma > 0 and (edge - fixed_sharpe) >= 2 * sigma
# Conjonction complete : significatif ET favorable. `dm_stat_median > 0` signifie que
# l'evolue a une perte PLUS ELEVEE que le facteur fixe -- significatif, mais defavorable.
dm_sign_ok = np.isfinite(dm_stat_median) and dm_stat_median < 0
dm_ok = np.isfinite(dm_p_median) and dm_p_median < 0.05 and dm_sign_ok

print("facteur fixe de reference : %-9s Sharpe OOS %.4f" % (best_fixed_name, fixed_sharpe))
print("champion evolue           : Sharpe OOS %.4f  (sigma inter-seeds %.4f)" % (edge, sigma))
print("edge - baseline           : %.4f   >= 2 sigma ? %s" % (edge - fixed_sharpe, sigma_ok))
print("Diebold-Mariano p median  : %.4f   < 0.05 ?      %s" % (dm_p_median, dm_p_median < 0.05))
print("Diebold-Mariano stat med. : %+.4f  < 0 (favorable) ? %s" % (dm_stat_median, dm_sign_ok))

if sigma_ok and dm_ok:
    verdict = "BEATS"
elif (not sigma_ok) and (not dm_ok):
    verdict = "NO BEATS"
else:
    verdict = "INCONCLUSIVE"
print("\nVERDICT :", verdict)

facteur fixe de reference : lowvol    Sharpe OOS -0.6359
champion evolue           : Sharpe OOS -0.1822  (sigma inter-seeds 0.2430)
edge - baseline           : 0.4537   >= 2 sigma ? False
Diebold-Mariano p median  : 0.0003   < 0.05 ?      True
Diebold-Mariano stat med. : -1.8227  < 0 (favorable) ? True

VERDICT : INCONCLUSIVE


In [11]:
# Robustesse au choix de la reference. La conjonction §C depend du facteur fixe auquel on se
# compare : un verdict qui ne tient que sous un choix de reference n'est pas un verdict, c'est
# un artefact de selection. On recalcule donc les deux jambes pour les QUATRE facteurs et on
# publie la table entiere, plutot que le seul chiffre du facteur retenu.
robust = []
for name, sig in FIXED.items():
    fl = np.concatenate([oos_losses(sig[te0:te1], fwd.values[te0:te1])
                         for _, _, te0, te1 in EVAL_FOLDS])
    ps, ss = [], []
    for r in rows:
        L = min(len(r["_loss"]), len(fl))
        st, p = dm_test_hac(r["_loss"][:L], fl[:L])
        ss.append(st); ps.append(p)
    p_med, s_med = float(np.nanmedian(ps)), float(np.nanmedian(ss))
    dm_ok_f = np.isfinite(p_med) and p_med < 0.05 and np.isfinite(s_med) and s_med < 0
    fs = baseline.loc[name, "Sharpe OOS (comparaison)"]
    s_ok_f = np.isfinite(sigma) and sigma > 0 and (edge - fs) >= 2 * sigma
    robust.append({"facteur": name, "Sharpe OOS compar.": fs, "edge": edge - fs,
                   ">= 2 sigma": s_ok_f, "DM p med": p_med, "DM stat med": s_med,
                   "conjonction": bool(s_ok_f and dm_ok_f)})
robust = pd.DataFrame(robust).set_index("facteur").round(4)
print("conjonction §C, facteur de reference par facteur :")
robust

conjonction §C, facteur de reference par facteur :


,Sharpe OOS compar.,edge,>= 2 sigma,DM p med,DM stat med,conjonction
facteur,,,,,,
momentum,1.5247,-1.7069,False,0.0085,-2.3570,False
reversal,-0.8912,0.7090,True,0.0069,-2.0056,True
lowvol,-0.6359,0.4537,False,0.0002,-1.8227,False
volume,1.0793,-1.2615,False,0.1057,0.5090,False


In [12]:
# Temoin negatif du critere directionnel. Deux series de pertes ou le modele "evolu"
# est significativement MOINS precis que le fixe : la difference est reelle et
# significative (p < 0.05), mais elle est defavorable. Le critere retenu doit la refuser,
# l'ancien (p < 0.05 seul) l'aurait acceptee comme une victoire.
rng_w = np.random.RandomState(0)
_l_fixe = rng_w.normal(0.0, 1.0, 400)
_l_evol = _l_fixe + 0.25 + rng_w.normal(0.0, 0.5, 400)   # perte systematiquement plus elevee
_w_stat, _w_p = dm_test_hac(_l_evol, _l_fixe)
print("temoin defavorable : DM stat %+.4f   p %.4f" % (_w_stat, _w_p))
print("  ancien critere  (p < 0.05 seul)          : %s" % (_w_p < 0.05))
print("  critere retenu  (p < 0.05 ET stat < 0)   : %s" % (_w_p < 0.05 and _w_stat < 0))
assert _w_p < 0.05 and _w_stat > 0, "le temoin doit etre significatif ET defavorable"
assert not (_w_p < 0.05 and _w_stat < 0), "le critere directionnel doit refuser ce cas"
print("temoin negatif OK : significatif, defavorable, et refuse par le critere retenu.")

temoin defavorable : DM stat +7.7405   p 0.0000
  ancien critere  (p < 0.05 seul)          : True
  critere retenu  (p < 0.05 ET stat < 0)   : False
temoin negatif OK : significatif, defavorable, et refuse par le critere retenu.


## 6. Lecture du verdict

Le verdict ci-dessus est produit par la **conjonction** exigee par la doctrine §C, pas par une
impression. Trois cas, et ce qu'ils veulent dire :

- **BEATS** - l'ecart au facteur fixe depasse 2 sigma inter-seeds **et** Diebold-Mariano est
  significatif **et favorable** (statistique negative : l'evolue est plus precis). Une difference
  significative mais defavorable n'ouvre pas cette branche : c'est le troisieme terme ajoute par la
  cellule 5.3.
- **NO BEATS** - aucune des conditions ne tient. Resultat honnete, et le plus frequent sur un panier
  de 10 actifs : l'evolution sur-ajuste le bloc d'entrainement.
- **INCONCLUSIVE** - une partie seulement de la conjonction tient. C'est le piege que la doctrine
  nomme : un `+N sigma` avec un `p` non significatif n'est **pas** une victoire ; un `p`
  significatif avec un signe defavorable n'en est pas une non plus.

### Le verdict depend du facteur de reference - et la table le montre

La cellule precedente publie la conjonction pour les **quatre** facteurs fixes, pas seulement pour
celui retenu : un verdict qui ne tient que sous un choix de reference n'est pas un verdict, c'est un
artefact de selection. Ici elle ne tient que pour `reversal`, qui est justement le **plus faible**
des quatre hors-echantillon - autrement dit, le seul facteur que le champion evolue bat est celui
dont la baseline est la moins bonne.

La reference retenue est choisie sur l'**IC d'entrainement**, sans regarder la fenetre d'evaluation ;
sous ce choix, la conjonction echoue sur la jambe sigma et le verdict est `INCONCLUSIVE`. C'est le
verdict publie, et la table donne au lecteur de quoi le contester.

### Le biais, separe de la precision

Un modele peut « gagner » parce qu'il est **biaise** dans le bon sens, pas parce qu'il est plus
**precis**. La cellule suivante separe les deux, comme l'exige §C.


In [13]:
def bias_report(sig, target, te0, te1):
    s = signal_to_rank(sig[te0:te1])
    t = pd.DataFrame(target[te0:te1]).rank(axis=1, pct=True).values
    pred = (s - 0.5) * 2.0
    e = pred - t
    return float(np.nanmean(e)), float(np.nanstd(e))


acc = []
for label, sig in [("evolue (seed 0)", evaluate(champions[SEEDS[0]])),
                   ("fixe: %s" % best_fixed_name, best_fixed)]:
    bs, ss = [], []
    for _, _, te0, te1 in FOLDS[1:]:
        b, s = bias_report(sig, fwd.values, te0, te1)
        bs.append(b); ss.append(s)
    acc.append({"modele": label, "biais signe moyen": np.nanmean(bs),
                "ecart-type erreur": np.nanmean(ss)})

pd.DataFrame(acc).set_index("modele").round(4)

,biais signe moyen,ecart-type erreur
modele,,
evolue (seed 0),-0.44,0.6584
fixe: lowvol,-0.45,0.6323


## 7. Exercices

Trois exercices, dans l'ordre de difficulte. Ils ne bloquent pas l'execution du carnet : les
cellules ci-dessous s'executent telles quelles meme non completees.

### Exercice 1 - enrichir le jeu de primitives

Ajoutez un operateur `corr20` (correlation glissante 20 jours entre le signal courant et un
second signal) au dictionnaire `PRIMITIVES`, en respectant l'aritie 2. Verifiez que
`random_expr` peut le tirer et que `evaluate` l'evalue sans `None`.

### Exercice 2 - changer la cible

Le carnet predit le rendement du **lendemain**. Remplacez la cible par le rendement cumule sur
5 jours, puis relancez l'evolution. L'IC monte-t-il, et le verdict change-t-il ? Attention : la
fenetre de prediction plus longue reduit le nombre d'observations independantes.

### Exercice 3 - mesurer la sur-ajustement

Comparez l'IC **d'entrainement** du champion a son IC **hors-echantillon**. Un ecart large est la
signature du sur-ajustement. Proposez une modification du fitness qui penalise cet ecart.

In [14]:
# --- Exercice 1 : ajouter une primitive -------------------------------------
# TODO etudiant : definir corr20(x, y) puis l'enregistrer dans PRIMITIVES.
def corr20(x, y):
    # Indice : pd.DataFrame(x).rolling(20).corr(pd.Series(...)) ne diffuse pas directement ;
    # calculer la correlation glissante ligne a ligne sur les deux panneaux.
    return None  # TODO etudiant : remplacer par le calcul


# --- Exercice 2 : changer la cible ------------------------------------------
fwd_5 = close.pct_change(5).shift(-5).reindex(ret.index)  # TODO etudiant : ne pas oublier l'anti-fuite
print("cible 5 jours disponible :", fwd_5.shape)


# --- Exercice 3 : ecart train / hors-echantillon -----------------------------
def gap_train_oos(individual):
    # TODO etudiant : rendre (IC train, IC OOS) du meme individu sur les memes blocs.
    return None  # TODO etudiant


print("Exercices a completer : 1, 2, 3")

cible 5 jours disponible : (1643, 10)
Exercices a completer : 1, 2, 3


## 8. Ce que ce carnet ne fait pas

- **Pas de mini-boucle LLM.** La comparaison « evolution vs LLM » demande un acces modele et un
  budget d'evaluation identique pour les deux approches ; elle est suivie separement (#20041).
- **Pas de backtest QuantConnect.** Le carnet mesure un signal de recherche (IC, portefeuille
  long-short jouet) ; le passage a une strategie LEAN est un autre travail, avec ses couts.
- **Pas de couts de transaction** dans le Sharpe affiche : un alpha a fort turnover y perdrait
  l'essentiel de son edge. C'est une limite **declaree**, pas un oubli.

## References

- Li, Yang, Yang, Xu, Wang, Liu, Bian - *R&D-Agent-Quant: A Multi-Agent Framework for Data-Centric
  Factors and Model Joint Optimization*, Microsoft Research Asia, arXiv:2505.15155. Archive au
  gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Trading\`.
- Serie `Search/Part1-Foundations/Search-05-GeneticAlgorithms.ipynb` - l'organe evolutionnaire natif.
- `.claude/rules/pr-review-discipline.md` §C - la doctrine de validation appliquee comme fitness :
  walk-forward **5 blocs**, **4 seeds**, Diebold-Mariano sur une perte de precision (`mse`) et
  **conjonction avec le signe**. Le carnet applique §C ; il ne pretend pas faire plus.

Issue de suivi : #20041 (Part of #19306).